# Objective and PosteriorTransform Workflow

multi-output surrogateを単一のdecision utilityへ接続するとき、`PosteriorTransform`とMC `Objective`は同じものではありません。

```text
X
 ↓
multi-output posterior
 ├─ PosteriorTransform → transformed posterior → analytic / MC acquisition
 └─ sampling → posterior samples → MC Objective → MC acquisition
```

このNotebookではBoTorch native objectsをrobotorchan modelへ直接渡し、candidate生成まで確認します。

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.analytic import LogExpectedImprovement
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.objective import (
    GenericMCObjective,
    LinearMCObjective,
    ScalarizedPosteriorTransform,
)
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Two-output surrogate

2出力を予測する`SingleTaskGP`を作ります。ここではmulti-objective Pareto optimizationではなく、2出力から**scalar decision utility**を作る問題です。

In [ ]:
def outcomes(X: torch.Tensor) -> torch.Tensor:
    first = torch.sin(2 * torch.pi * X)
    second = torch.cos(2 * torch.pi * X) + 0.2 * X
    return torch.cat([first, second], dim=-1)


train_X = torch.linspace(0.05, 0.95, 10).unsqueeze(-1)
train_Y = outcomes(train_X)
bounds = torch.tensor([[0.0], [1.0]])

model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()

probe_X = torch.tensor([[0.25], [0.75]])
with torch.no_grad():
    raw_posterior = model.posterior(probe_X)
print("raw posterior mean shape:", raw_posterior.mean.shape)

## 3. PosteriorTransform acts before sampling

線形scalarizationなら`ScalarizedPosteriorTransform`でmulti-output posteriorそのものをscalar posteriorへ変換できます。meanだけでなくcovarianceも線形写像後のposteriorとして扱われます。

In [ ]:
weights = torch.tensor([0.7, 0.3])
posterior_transform = ScalarizedPosteriorTransform(weights=weights)

with torch.no_grad():
    transformed = model.posterior(
        probe_X,
        posterior_transform=posterior_transform,
    )
    expected = posterior_transform(raw_posterior)

torch.testing.assert_close(transformed.mean, expected.mean)
torch.testing.assert_close(transformed.variance, expected.variance)
print("transformed posterior mean shape:", transformed.mean.shape)

## 4. Analytic acquisition after PosteriorTransform

posteriorがscalarになったため、compatibleなanalytic acquisitionへ接続できます。

In [ ]:
scalar_train_Y = train_Y @ weights
analytic_acq = LogExpectedImprovement(
    model=model,
    best_f=scalar_train_Y.max(),
    posterior_transform=posterior_transform,
)
analytic_candidate, analytic_value = optimize_acqf(
    acq_function=analytic_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("PosteriorTransform candidate:", analytic_candidate)
print("LogEI value:", analytic_value)

## 5. MC Objective acts on posterior samples

`LinearMCObjective`はposterior distributionを変換するのではなく、sampling後の`[..., q, m]` samplesをscalar utility `[..., q]`へ変換します。

In [ ]:
sampler = SobolQMCNormalSampler(torch.Size([128]), seed=123)
linear_objective = LinearMCObjective(weights=weights)

with torch.no_grad():
    samples = sampler(model.posterior(probe_X))
    objective_samples = linear_objective(samples)
print("posterior samples shape:", samples.shape)
print("objective samples shape:", objective_samples.shape)

## 6. MC acquisition with the linear objective

In [ ]:
mc_acq = qLogExpectedImprovement(
    model=model,
    best_f=scalar_train_Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=321),
    objective=linear_objective,
)
mc_candidate, mc_value = optimize_acqf(
    acq_function=mc_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("LinearMCObjective candidate:", mc_candidate)
print("qLogEI value:", mc_value)

## 7. Nonlinear utility belongs on the MC-objective path

非線形utilityは一般にGaussian posteriorをGaussianのまま保つとは限りません。ここでは`GenericMCObjective`でsamples上のutilityを定義します。

In [ ]:
def nonlinear_utility(Y: torch.Tensor, X: torch.Tensor | None = None) -> torch.Tensor:
    del X
    return Y[..., 0] - 0.25 * Y[..., 1].square()


nonlinear_objective = GenericMCObjective(nonlinear_utility)
baseline_samples = SobolQMCNormalSampler(
    torch.Size([128]),
    seed=456,
)(model.posterior(train_X))
nonlinear_best_f = nonlinear_objective(baseline_samples).mean(dim=0).max()

nonlinear_acq = qLogExpectedImprovement(
    model=model,
    best_f=nonlinear_best_f,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=654),
    objective=nonlinear_objective,
)
nonlinear_candidate, nonlinear_value = optimize_acqf(
    acq_function=nonlinear_acq,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=64,
)
print("nonlinear-objective candidate:", nonlinear_candidate)
print("qLogEI value:", nonlinear_value)

## 8. Candidate evaluation in raw outcome space

optimizerが返すのは入力`X`です。実験・simulationではraw outcomesを観測し、その後decision utilityを計算します。

In [ ]:
candidate_X = nonlinear_candidate
candidate_Y = outcomes(candidate_X)
candidate_utility = nonlinear_utility(candidate_Y)
updated_X = torch.cat([train_X, candidate_X], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

print("candidate X:", candidate_X)
print("raw candidate Y:", candidate_Y)
print("decision utility:", candidate_utility)
print("updated shapes:", updated_X.shape, updated_Y.shape)

## 9. Visualize the different scalar utilities

In [ ]:
grid = torch.linspace(0.0, 1.0, 300).unsqueeze(-1)
grid_Y = outcomes(grid)
linear_utility = grid_Y @ weights
nonlinear_values = nonlinear_utility(grid_Y)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid.squeeze(-1), linear_utility, label="linear utility")
ax.plot(grid.squeeze(-1), nonlinear_values, label="nonlinear utility")
ax.scatter(
    analytic_candidate.item(),
    (outcomes(analytic_candidate) @ weights).item(),
    marker="x",
    s=80,
    label="PosteriorTransform candidate",
)
ax.scatter(
    nonlinear_candidate.item(),
    nonlinear_utility(outcomes(nonlinear_candidate)).item(),
    marker="x",
    s=80,
    label="nonlinear MC candidate",
)
ax.legend()
plt.show()

## 10. Responsibility map

| Concern | Correct layer |
| --- | --- |
| affine transform of posterior distribution | `PosteriorTransform` |
| scalar utility of posterior samples | `MCAcquisitionObjective` |
| nonlinear sample utility | `GenericMCObjective` |
| Pareto / hypervolume vector objective | multi-objective acquisition path |
| unknown outcome feasibility | acquisition outcome constraints |
| known feasibility of X | acquisition optimizer constraints |
| candidate input perturbations | uncertainty / scenario generation |
| risk across environmental scenarios | risk aggregation after objective evaluation |

## 11. Important boundaries

- `PosteriorTransform`とMC `Objective`をAPI統一のために同一抽象化しません。
- native BoTorch objectsをrobotorchan-specific wrapperへ変換しません。
- nonlinear sample utilityを`PosteriorTransform`として偽装しません。
- qLogEHVI/qLogNEHVIではPareto vectorを保持するため、scalarizationがdefaultではありません。
- `ScalarizedPosteriorTransform`対応はmodel posterior contractに依存します。ALEBO等には明示的restrictionがあります。
- output reduction中のspecialized modelではoriginal output space復元との順序が重要です。
- robust input perturbationとrisk aggregationはPhase 21のrobust workflowで扱います。

## 12. Related documentation

- `docs/development/objective-posterior-transform-audit.md`
- `docs/theory/acquisition/01_foundations.md`
- `docs/theory/07_multitask_multioutput.md`
- `27_multiobjective_acquisition.ipynb`

Objectiveは「posterior outputsを何のutilityとして使うか」、PosteriorTransformは「posterior distributionをどう変換するか」という異なる責務を持ちます。